# Before you train: audit a pick-and-place dataset

Use this notebook to inspect pick-and-place demonstrations before training or sharing a recording. Run the example first, then try it with your own compatible LeRobot dataset.

The public **cube-to-bowl** example was recorded on an SO-100 in LeRobot v2.1. You can also point the notebook at a local LeRobot dataset; the local workflow has been tested with a separate v3 recording.

**Runtime → Run all** runs the public example on CPU (about 66 MB, no account). Nothing is published or edited. Keep private or restricted data in an approved local Jupyter environment, not a hosted Colab session.

Demo checkpoint: 9 pass, 1 review (episode 000008, gripper camera). Compare sharpness across episodes **within** the same camera; the absolute values are not a cross-camera grade.

In [ ]:
# @title Install the grader and plotting tools
%pip install -q "kalanos[numeric,video]==0.10.1" "huggingface-hub==1.33.0" "matplotlib==3.11.2"

In [ ]:
from pathlib import Path

# Leave blank for the public demo. Otherwise give the directory containing meta/info.json.
LOCAL_DATASET = ""  # @param {type:"string"}
DEMO_ID = "triton7777/cube_to_bowl_so100"
DEMO_REVISION = "9f490c4e624034eb5c6e961e65767abb9ab7c391"
PROFILE = "vision-imitation-v1"

In [ ]:
# @title Open the local data or download the pinned public snapshot
import json

from huggingface_hub import snapshot_download

if LOCAL_DATASET.strip():
    dataset = Path(LOCAL_DATASET.strip()).expanduser().resolve()
    source_label = dataset.name
else:
    dataset = Path(snapshot_download(
        repo_id=DEMO_ID, repo_type="dataset", revision=DEMO_REVISION, token=False
    ))
    source_label = f"{DEMO_ID} @ {DEMO_REVISION[:8]}"

manifest = dataset / "meta" / "info.json"
if not manifest.is_file():
    raise FileNotFoundError("Choose the unpacked LeRobot dataset root containing meta/info.json.")
info = json.loads(manifest.read_text(encoding="utf-8"))
print(f"{source_label} | LeRobot {info.get('codebase_version')}")
print(f"Declared episodes: {info.get('total_episodes')} | FPS: {info.get('fps')}")

In [ ]:
# @title Run one local audit and save audit.json
import os
import subprocess
import tempfile

# A fresh directory prevents an old report from masking a failed run.
output = Path(tempfile.mkdtemp(prefix="kalanos-audit-", dir=Path.cwd()))
audit_json = output / "audit.json"
command = [
    "kalanos", "grade", str(dataset), "--tier", "full",
    "--profile", PROFILE, "--report", str(audit_json),
    "--fail-on", "blocked,review,unknown",
]
result = subprocess.run(command, capture_output=True, text=True, env={**os.environ, "NO_COLOR": "1"})
(output / "grader.log").write_text(result.stdout + "\n" + result.stderr, encoding="utf-8")
# Exit 1 can be a gate decision. Require a new usable audit before plotting.
if result.returncode not in (0, 1) or not audit_json.is_file():
    raise RuntimeError(f"Grading did not produce a report (exit {result.returncode}).\n{result.stderr[-2000:]}")
print("Saved audit.json.")

In [ ]:
from collections import Counter

import kalanos

# The public loader reads compact keys and resolves reason references.
report = kalanos.load_report(audit_json)
if not isinstance(report, kalanos.Report) or not report.episodes:
    raise RuntimeError("No episode results. Inspect grader.log and the dataset layout.")

status_order = ("pass", "review", "blocked", "unknown")
colors = {"pass": "#2c7659", "review": "#b95514", "blocked": "#bb3046", "unknown": "#64748b"}

def decision(episode):
    return episode.eligibility.status.value if episode.eligibility else "unknown"

counts = Counter(decision(episode) for episode in report.episodes)
readiness = report.readiness.score if report.readiness else None
print(f"Loaded episodes: {len(report.episodes)}")
print(" | ".join(f"{label.title()}: {counts[label]}" for label in status_order))
print("Training readiness:", "Not established" if readiness is None else f"{readiness:.1f}/100")
if report.readiness:
    for reason in report.readiness.reasons:
        print("-", reason)

In [ ]:
import matplotlib.pyplot as plt

# Use the reported decisions, not thresholds invented from descriptive scores.
fig, ax = plt.subplots(figsize=(8, 3.5), layout="constrained")
bars = ax.bar(status_order, [counts[s] for s in status_order], color=[colors[s] for s in status_order])
ax.bar_label(bars, padding=3)
ax.set(title="Which demonstrations need attention?", ylabel="Episodes", ylim=(0, max(counts.values()) * 1.20 + 1))
ax.spines[["top", "right"]].set_visible(False)
fig.savefig(output / "episode-decisions.png", dpi=150)
plt.show()

In [ ]:
# @title Plot measured camera sharpness by episode
from matplotlib.patches import Patch


points = []
for episode in report.episodes:
    for stream in episode.streams:
        metric = stream.metrics.get("sharpness_score")
        if metric is not None and metric.value is not None:
            field = stream.source_field or "Camera (source field not recorded)"
            points.append((episode.id, field, float(metric.value)))

# Missing measurements are not passes and are never plotted as zero.
if not points:
    print("No camera sharpness measurements. Check the report scope and coverage.")
else:
    position = {episode.id: idx for idx, episode in enumerate(report.episodes)}
    cameras = sorted({field for _, field, _ in points})
    # Cool hues and distinct shapes keep cameras apart from the orange decision colour.
    camera_colors, camera_markers = ["#3b6fa0", "#7a5fa5", "#4f8a8b"], ["o", "s", "^"]
    fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
    for n, camera in enumerate(cameras):
        measured = [(position[eid], value) for eid, field, value in points if field == camera]
        # A small offset keeps two cameras on the same episode from overlapping.
        shift = (n - (len(cameras) - 1) / 2) * 0.16
        ax.scatter(
            [x + shift for x, _ in measured], [y for _, y in measured],
            label=camera, color=camera_colors[n % 3], marker=camera_markers[n % 3],
            s=45, zorder=3,
        )
    # Shade episodes with a review or blocked decision; inspect the reasons for the cause.
    flagged_positions = {position[e.id] for e in report.episodes if decision(e) in {"review", "blocked"}}
    for idx in sorted(flagged_positions):
        ax.axvspan(idx - 0.4, idx + 0.4, color=colors["review"], alpha=0.12, zorder=0)
    if len(report.episodes) <= 16:
        ax.set_xticks(list(position.values()), [eid.rsplit("_", 1)[-1] for eid in position], rotation=35)
    ax.set(title="Camera sharpness by episode", xlabel="Report episode (order if unlabeled)", ylabel="Sharpness score")
    handles, labels = ax.get_legend_handles_labels()
    if flagged_positions:
        handles.append(Patch(color=colors["review"], alpha=0.12))
        labels.append("Review or blocked episode")
    ax.legend(handles, labels, fontsize="small")
    ax.spines[["top", "right"]].set_visible(False)
    fig.savefig(output / "camera-sharpness.png", dpi=150)
    plt.show()

In [ ]:
# @title Inspect one flagged episode and find the audit files
from IPython.display import FileLink, display

flagged = next((e for e in report.episodes if decision(e) in {"review", "blocked"}), None)
if flagged is None:
    print("No review or blocked episode. Inspect a coverage gap or spot-check a pass.")
else:
    print(f"{flagged.id} | {decision(flagged)}")
    for reason in (flagged.eligibility.reasons or [])[:6]:
        print(f"- {reason.id}: {reason.detail or 'reason text unavailable'}")

for name in ("audit.json", "episode-decisions.png", "camera-sharpness.png", "grader.log"):
    path = output / name
    if path.is_file():
        display(FileLink(str(path.relative_to(Path.cwd()))))

## Make a dataset review note

Save the audit and decision graph with your dataset review. In Colab, find them in the Files panel under the new `kalanos-audit-` folder.

1. Which episode and check did you inspect? Compare the reported evidence with what you saw in a replay, or identify a coverage gap.
2. Did you keep, correct, or recollect the episode? Explain why. Keep the original audit if you rerun on changed data.
3. For stacking, manually check that demonstrations cover different stages. The audit does not measure task success or replace policy rollouts.

**Continue with Kalanos:** [Browse dataset reports](https://hub.kalanos.ai) · [Use the grader locally](https://github.com/KalanosAI/kalanos) · [Attach an audit to a training run](https://github.com/KalanosAI/kalanos-tracking)